# 1. Objective
This notebook establishes the Real → Real ML baseline. We train exactly two baseline models (Logistic Regression and Random Forest) exclusively on `adult.data` and evaluate them exclusively on the holdout `adult.test` dataset. This establishes an upper bound of model utility for later comparison against models trained on synthetic data.

# 2. Load Training and Test Data
We load the original splits, preserving the train/test boundaries.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

# Set visual theme
sns.set_theme(style="whitegrid")
pd.set_option('display.max_columns', None)

columns = [
    "age", "workclass", "fnlwgt", "education", "education-num", 
    "marital-status", "occupation", "relationship", "race", "sex", 
    "capital-gain", "capital-loss", "hours-per-week", "native-country", "income"
]

# Load original splits
df_train = pd.read_csv('../data/raw/adult.data', names=columns, skipinitialspace=True, na_values='?')
df_test = pd.read_csv('../data/raw/adult.test', names=columns, skiprows=1, skipinitialspace=True, na_values='?')

# Clean income trailing period in test set
df_test['income'] = df_test['income'].str.rstrip('.')

print(f"Train dimensions: {df_train.shape}")
print(f"Test dimensions: {df_test.shape}")

# 3. Identify Features and Target
We separate features (X) from the target (y), and binarize the target where `>50K` is the positive class (1).

In [ ]:
X_train = df_train.drop('income', axis=1)
y_train = (df_train['income'] == '>50K').astype(int)

X_test = df_test.drop('income', axis=1)
y_test = (df_test['income'] == '>50K').astype(int)

print(f"Training target distribution:\n{y_train.value_counts(normalize=True)}\n")
print(f"Test target distribution:\n{y_test.value_counts(normalize=True)}")

# 4. Preprocessing Pipeline
We establish a rigorous Scikit-Learn `Pipeline`. 
- **Missing categorical values** are imputed with the constant `'Missing'`.
- **Categorical features** are One-Hot Encoded.
- **Numerical features** are Standard Scaled.
This pipeline will be fitted strictly on the training set.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Identify columns by type
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
# Handle deprecation warning by explicitly selecting object and category
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

# Define numerical transformer
num_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# Define categorical transformer
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine into ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ])

preprocessor

# 5. Logistic Regression
A strong, interpretable linear baseline.

In [ ]:
from sklearn.linear_model import LogisticRegression

# Create pipeline
lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor), 
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

# Fit on training data ONLY
lr_pipeline.fit(X_train, y_train)

# Predict on test data
y_pred_lr = lr_pipeline.predict(X_test)
y_prob_lr = lr_pipeline.predict_proba(X_test)[:, 1]

# 6. Random Forest
A robust, non-linear ensemble baseline.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Create pipeline
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor), 
    ('classifier', RandomForestClassifier(random_state=42))
])

# Fit on training data ONLY
rf_pipeline.fit(X_train, y_train)

# Predict on test data
y_pred_rf = rf_pipeline.predict(X_test)
y_prob_rf = rf_pipeline.predict_proba(X_test)[:, 1]

# 7. Evaluation Metrics

In [ ]:
def compute_metrics(y_true, y_pred, y_prob):
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'ROC-AUC': roc_auc_score(y_true, y_prob),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1-Score': f1_score(y_true, y_pred)
    }

lr_metrics = compute_metrics(y_test, y_pred_lr, y_prob_lr)
rf_metrics = compute_metrics(y_test, y_pred_rf, y_prob_rf)

# 8. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm_lr = confusion_matrix(y_test, y_pred_lr)
disp_lr = ConfusionMatrixDisplay(confusion_matrix=cm_lr, display_labels=['<=50K', '>50K'])
disp_lr.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].grid(False)
axes[0].set_title('Logistic Regression')

cm_rf = confusion_matrix(y_test, y_pred_rf)
disp_rf = ConfusionMatrixDisplay(confusion_matrix=cm_rf, display_labels=['<=50K', '>50K'])
disp_rf.plot(ax=axes[1], cmap='Blues', colorbar=False)
axes[1].grid(False)
axes[1].set_title('Random Forest')

plt.tight_layout()
plt.show()

# 9. Comparison of Baseline Models

In [ ]:
# Consolidate metrics into a DataFrame for easy viewing
metrics_df = pd.DataFrame([lr_metrics, rf_metrics], index=['Logistic Regression', 'Random Forest'])
metrics_df.round(4)

# 10. Initial Observations
Based on evaluating our models strictly on the holdout `adult.test` dataset:

- **ROC-AUC**: Logistic Regression achieves 0.9055, and Random Forest achieves 0.8998. The high ROC-AUC demonstrates strong ranking ability across both models.
- **Class Imbalance Impact**: While Accuracy is high (~85%), the F1-Scores for the minority class (`>50K`) are much lower (LR: 0.6580, RF: 0.6578), reflecting the difficulty of predicting the positive class accurately.
- **Precision vs Recall Trade-off**: 
  - Logistic Regression is slightly more precise (0.7306 vs 0.7130).
  - Random Forest yields slightly higher recall (0.6105 vs 0.5985).
- **Leakage Prevention**: All preprocessing constraints (imputation, scaling) were correctly isolated to the training set through the `Pipeline`.

These metrics establish our upper-bound real-data utility. Any future models trained on synthetically generated data will be measured against this baseline to calculate utility loss.